# Relation Extraction with PEFT (LoRA) 🤗

In this notebook, we will learn how to fine-tune a pre-trained transformer model (like `bert-base-uncased`) for **Relation Extraction** using Parameter-Efficient Fine-Tuning (PEFT) and Low-Rank Adaptation (LoRA).

### What is Relation Extraction?
Relation Extraction (RE) is a core task in Natural Language Processing (NLP) that aims to identify semantic relationships between entities in a sentence. For example, in the sentence:
"The entity <e1>author</e1> of a <e2>treatise</e2> is..."
We want to classify the relationship between "author" (entity 1) and "treatise" (entity 2) as `Writer-Work(e1,e2)`.

### Method:
1. We will use the **SemEval-2010 Task 8** dataset.
2. We will insert special entity markers `<e1>`, `</e1>`, `<e2>`, `</e2>` to demarcate the target entities. We will add these markers as special tokens to our tokenizer and model.
3. We will model this as a **Sequence Classification** task (using `AutoModelForSequenceClassification`).
4. We will wrap the model using `peft` and train only the LoRA adapters + sequence classification head, keeping the base model weights frozen.

In [ ]:
# Install dependencies
!pip install -q datasets evaluate transformers accelerate scikit-learn peft

In [ ]:
import evaluate
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    pipeline,
)
from peft import LoraConfig, TaskType, get_peft_model, PeftModel

## 1. Load and Inspect the Dataset

We load SemEval-2010 Task 8. To prevent security restrictions from blocking remote Python dataset loading scripts in newer versions of the `datasets` library, we load from the repository's auto-converted **Parquet branch** (`revision="refs/convert/parquet"`).

In [ ]:
dataset = load_dataset("joelniklaus/sem_eval_2010_task_8", revision="refs/convert/parquet")
print(dataset)

In [ ]:
# Inspect an example training instance
example = dataset["train"][0]
print("Sentence:", example["sentence"])
print("Relation ID:", example["relation"])

# Get class mappings
relation_feature = dataset["train"].features["relation"]
label_names = relation_feature.names
num_labels = len(label_names)
print(f"\nRelation Class Name: {label_names[example['relation']]}")
print(f"Total classes: {num_labels}")

## 2. Tokenizer Setup with Special Entity Markers

We load the tokenizer and add `<e1>`, `</e1>`, `<e2>`, `</e2>` as special tokens. Adding them explicitly ensures they are not broken down into subwords during tokenization, allowing the transformer to learn dedicated representations for entity boundaries.

In [ ]:
model_checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

special_tokens_dict = {"additional_special_tokens": ["<e1>", "</e1>", "<e2>", "</e2>"]}
num_added_toks = tokenizer.add_special_tokens(special_tokens_dict)
print(f"Added {num_added_toks} special tokens to the tokenizer.")

## 3. Preprocess Dataset

In [ ]:
def preprocess_function(examples):
    result = tokenizer(examples["sentence"], truncation=True, max_length=128)
    result["label"] = examples["relation"]
    return result

tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)
print(tokenized_dataset)

## 4. Initialize Model and PEFT (LoRA)

We load the sequence classification model, resize the token embeddings to accommodate our new special tokens, and configure LoRA.

In [ ]:
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in enumerate(label_names)}

model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

# Resize token embeddings to include new special tokens
model.resize_token_embeddings(len(tokenizer))
print("Resized model embeddings.")

In [ ]:
# Define PEFT LoRA Config for Sequence Classification
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 5. Train the Model

We load evaluation metrics (accuracy and F1-score) and start training with the Hugging Face `Trainer`.

In [ ]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1_macro = f1_metric.compute(predictions=predictions, references=labels, average="macro")["f1"]
    f1_micro = f1_metric.compute(predictions=predictions, references=labels, average="micro")["f1"]
    return {
        "accuracy": acc,
        "f1_macro": f1_macro,
        "f1_micro": f1_micro
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./bert-finetuned-relation-extraction-lora",
    learning_rate=2e-4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=100,
    report_to="none"
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

## 6. Inference

Finally, we load the trained PEFT adapter and run inference on new, unseen sentences.

In [ ]:
# Save model adapter
model.save_pretrained("./bert-finetuned-relation-extraction-lora")
tokenizer.save_pretrained("./bert-finetuned-relation-extraction-lora")

In [ ]:
# Load for inference
base_model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)
base_model.resize_token_embeddings(len(tokenizer))
inference_model = PeftModel.from_pretrained(base_model, "./bert-finetuned-relation-extraction-lora")

classifier = pipeline(
    "text-classification", 
    model=inference_model, 
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

test_sentence = "The entity <e1>author</e1> of a <e2>treatise</e2> is anonymous."
result = classifier(test_sentence)
print(f"Sentence: {test_sentence}")
print(f"Predicted relation: {result[0]['label']} (score: {result[0]['score']:.4f})")